# 水稻病虫害检测 + RAG + Agentic Loop

该 Notebook 调用项目中的模块，不再把全部实现堆在单个单元格中。
模型来自公开 Hugging Face 权重，并非本项目训练。


In [ ]:
# 首次运行取消注释：
# %pip install -r ../requirements.txt


In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("项目根目录：", PROJECT_ROOT)


## 1. 模型自检

In [ ]:
from rice_agent.services.detector import RiceDiseaseDetector

detector = RiceDiseaseDetector()
model_info = detector.model_info()
print(json.dumps(model_info, ensure_ascii=False, indent=2))


## 2. 构建或加载 Chroma RAG

In [ ]:
from rice_agent.services.rag_store import RiceKnowledgeStore

knowledge_store = RiceKnowledgeStore()
knowledge_store.build_or_load()
print("Chroma目录：", knowledge_store.chroma_dir)


## 3. 独立测试 RAG

In [ ]:
rag_results = knowledge_store.search(
    question="这种病害有哪些典型表现和基础管理建议？",
    disease_code="leaf_blast",
    k=4,
)
print(json.dumps(rag_results, ensure_ascii=False, indent=2))


## 4. 独立测试 YOLO

将 `TEST_IMAGE` 改成你自己的水稻图片路径。


In [ ]:
TEST_IMAGE = PROJECT_ROOT / "uploads" / "test.jpg"

if TEST_IMAGE.is_file():
    detection_result = detector.detect(TEST_IMAGE)
    print(json.dumps(
        detection_result,
        ensure_ascii=False,
        indent=2,
        default=str,
    ))
else:
    print("请先放入图片：", TEST_IMAGE)


## 5. 测试 DeepSeek Tool Calling

In [ ]:
from langchain_deepseek import ChatDeepSeek
from rice_agent.agent.tools import search_rice_knowledge
from rice_agent.config import settings

if settings.deepseek_api_key:
    tool_llm = ChatDeepSeek(
        model=settings.deepseek_model,
        api_key=settings.deepseek_api_key,
        base_url=settings.deepseek_base_url,
        temperature=0,
    ).bind_tools([search_rice_knowledge])

    response = tool_llm.invoke(
        "请调用工具查询稻瘟病的典型表现。"
    )
    print(response.tool_calls)
else:
    print("尚未配置DEEPSEEK_API_KEY")


## 6. Agentic Loop

In [ ]:
from rice_agent.agent.loop import RiceDiseaseAgent

if settings.deepseek_api_key and TEST_IMAGE.is_file():
    agent = RiceDiseaseAgent(verbose=True)
    answer = agent.chat(
        f"请检测图片 {TEST_IMAGE}，并说明可能的病虫害、"
        "典型表现、基础管理建议和资料来源。"
    )
    print(answer)
else:
    print("请先配置API并准备测试图片。")


## 7. 不支持 Tool Calling 时使用确定性流程

In [ ]:
from rice_agent.direct_pipeline import analyze_image_direct

if TEST_IMAGE.is_file():
    direct_result = analyze_image_direct(
        image_path=str(TEST_IMAGE),
        question="请说明可能的病虫害和基础管理建议。",
    )
    print(direct_result["answer"])
